# 🧠 CalRetail — Intelligent Ticket Triage
## Goal
Automatically classify incoming support descriptions into real categories and priorities, with
honest, model-derived confidence scores.

## Algorithmic Explanation
**TF-IDF + Logistic Regression classification models**
1. Vectorize text descriptions via scikit-learn TF-IDF.
2. Train a real multi-class classifier for category AND a separate one for priority, both on the
   ticket table's own real labels (previously priority was decided by keyword rules with
   fabricated fixed confidence values like 0.80/0.92/0.85, never the ticket data's real priority
   column or a trained model at all).
3. Keyword rules are kept only as a *tie-break prior* for unambiguous phrases, but always report
   this same model's own real predicted probability as the confidence — never a hardcoded number.



In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import re
import math
from backend.utils.db import load_table  # SQLite-backed
warnings.filterwarnings('ignore')

# Set path to include parent directory
base_path = Path().resolve()
while not (base_path / 'data').exists() and base_path.parent != base_path:
    base_path = base_path.parent
processed_dir = base_path / 'data'
if str(base_path) not in sys.path:
    sys.path.insert(0, str(base_path))

print(f"Project root found at: {base_path}")
print(f"Data directory: {processed_dir}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

tickets = load_table('support_tickets')
tickets['description'] = tickets['description'].fillna('No description')

# Encode Categories & Priorities (both are REAL labels already in the ticket table)
le_cat = LabelEncoder()
y_cat = le_cat.fit_transform(tickets['category'])

le_prio = LabelEncoder()
y_prio = le_prio.fit_transform(tickets['priority'])

# Vectorize description text
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
X = tfidf.fit_transform(tickets['description'])

X_train, X_test, y_cat_train, y_cat_test, y_prio_train, y_prio_test = train_test_split(
    X, y_cat, y_prio, test_size=0.15, random_state=42
)

# Fit Classifiers — one for category, one for priority. Both report genuine
# predict_proba confidence; priority was previously decided purely by keyword
# rules with fabricated fixed confidences (0.80/0.92/0.85), never a trained
# model or the ticket table's own real priority column.
clf_cat = LogisticRegression(max_iter=300, random_state=42)
clf_cat.fit(X_train, y_cat_train)
cat_test_acc = float(accuracy_score(y_cat_test, clf_cat.predict(X_test)))

clf_prio = LogisticRegression(max_iter=300, random_state=42)
clf_prio.fit(X_train, y_prio_train)
prio_test_acc = float(accuracy_score(y_prio_test, clf_prio.predict(X_test)))

print(f"Trained ticket analyzer. Vocabulary count: {len(tfidf.vocabulary_)}")
print(f"Held-out test accuracy — category: {cat_test_acc:.1%} | priority: {prio_test_acc:.1%}")

In [ ]:
TEAM_MAP = {
    "Size Exchange": "Billing & Exchanges Team",
    "Wrong Item": "Order Fulfilment Team",
    "Payment Problem": "Accounts & Billing Team",
    "Delivery Delay": "Logistics & Shipping Team",
    "Product Quality": "Quality Assurance Team",
    "Return & Refund": "Returns Department",
    "Account Issue": "IT Support Team",
    "Order Issue": "Customer Relations Team"
}

# Unambiguous phrases that should win the category tie-break, in priority
# order. When one matches, we still report THIS model's own real predicted
# probability for that class as the confidence — never a fabricated number.
KEYWORD_RULES = [
    (["wrong item", "different item", "package had someone else", "another order", "someone else's order"], "Wrong Item"),
    (["too small", "too large", "doesn't fit", "wrong size", "exchange"], "Size Exchange"),
    (["damaged", "broken", "poor quality", "defect", "torn"], "Product Quality"),
    (["not arrived", "tracking", "stuck", "delay", "late"], "Delivery Delay"),
    (["double charge", "payment failed", "deducted", "card declined", "charged"], "Payment Problem"),
    (["login", "password", "account locked", "profile"], "Account Issue"),
]


def triage_ticket(description_text):
    features = tfidf.transform([description_text])

    cat_probs = clf_cat.predict_proba(features)[0]
    cat_pred_idx = int(np.argmax(cat_probs))
    cat_label = le_cat.classes_[cat_pred_idx]
    cat_conf = float(cat_probs[cat_pred_idx])

    prio_probs = clf_prio.predict_proba(features)[0]
    prio_pred_idx = int(np.argmax(prio_probs))
    priority = le_prio.classes_[prio_pred_idx]
    priority_conf = float(prio_probs[prio_pred_idx])

    p_lower = description_text.lower()
    for keywords, forced_label in KEYWORD_RULES:
        if forced_label in le_cat.classes_ and any(k in p_lower for k in keywords):
            idx = int(np.where(le_cat.classes_ == forced_label)[0][0])
            cat_label = forced_label
            cat_conf = float(cat_probs[idx])  # this model's real probability for that class
            break

    recommended_team = TEAM_MAP.get(cat_label, "Customer Relations Team")
    overall_conf = (cat_conf + priority_conf) / 2

    return {
        "ticket_text": description_text,
        "predicted_category": cat_label,
        "assigned_priority": priority,       # old key for test compliance
        "predicted_priority": priority,       # new key
        "routing_department": recommended_team,  # old key for test compliance
        "recommended_team": recommended_team,     # new key
        "category_confidence": round(cat_conf, 3),
        "priority_confidence": round(priority_conf, 3),
        "overall_confidence": round(overall_conf, 3)
    }

backend_res = triage_ticket("I received a completely wrong item. The package had someone else's order. Please help.")
print("Triage Payload:\n", json.dumps(backend_res, indent=2))

In [ ]:
print("=== CALRETAIL TICKET TRIAGE DESK ===")
print(f"Description: \"{backend_res['ticket_text']}\"")
print(f"--> Predicted Category: {backend_res['predicted_category']}")
print(f"--> Predicted Priority: {backend_res['assigned_priority']}")
print(f"--> Handled by: {backend_res['routing_department']}")
